# LinguoMT — FLEURS × SeamlessM4T-v2

| | |
|---|---|
| **Model** | SeamlessM4T-v2-Large (end-to-end S2TT) |
| **Dataset** | google/fleurs |
| **Languages** | Igbo · Yoruba · Swahili  *(Hausa excluded: not in SeamlessM4T S2TT list)* |
| **Papers** | All 5 papers |
| **DEBUG runtime** | ~10 min |
| **FULL runtime** | ~30–60 min |

Runs this single experiment, consolidates its outputs, then generates a Markdown
results report with tables, cross-model comparisons, SOTA analysis, and discussion.

> **Before running:** Runtime → Change runtime type → **T4 GPU** (or A100).
> **Quick start:** Configure Step 4, then **Runtime → Run all**.

## About This Experiment — FLEURS × SeamlessM4T-v2

**Model:** `facebook/seamless-m4t-v2-large` — a massively multilingual **end-to-end**
model that performs speech-to-text translation (S2TT), ASR, and text translation in
a single pass with no intermediate transcription step.

**Dataset:** `google/fleurs` — a large multilingual speech benchmark with clean, read-speech
utterances. More diverse in speaking style and recording conditions than African-Celtic.
FLEURS results are directly comparable to most published multilingual speech translation papers.

**Languages evaluated:** Igbo (ibo) · Yoruba (yor) · Swahili (swh)
*(Hausa excluded: `hau` is not in SeamlessM4T's supported S2TT language list.
Swahili substituted to maintain 3-language coverage.)*

**What it tests:**
Zero-shot S2TT quality on the FLEURS benchmark — the standard evaluation set used by
most published multilingual speech translation systems. Results here are directly comparable
to published SOTA numbers. Also enables Igbo/Yoruba/Swahili cross-lingual analysis for Paper 5.

**Expected results:**
- BLEU scores typically higher than African-Celtic because FLEURS utterances are cleaner.
- Swahili outperforms Igbo and Yoruba — much more multilingual training data exists.
- Published SOTA on Swahili FLEURS S2TT is roughly 20–30 BLEU; Igbo/Yoruba are below 15.
- ASR WER: Swahili < 25%, Yoruba 30–50%, Igbo 40–60%.

**Comparison partner:**
Run `FLEURS__WhisperNLLB/notebooks/run_on_colab.ipynb` for the cascade baseline.
Results from all 4 notebooks together cover the full 2×2 grid for Papers 1–5.

---

## Per-paper configuration recipes

Choose the recipe that matches your paper, then copy the values into Step 4.

### Paper 1 — LinguoMT: Zero-shot benchmarking

```python
PAPER_MODE        = "benchmark"
DEBUG_MODE        = False
ENABLE_FINETUNING = False
SCALING_BUDGETS   = []
SOTA_FILE         = "sota/paper1_benchmark/sota_results.csv"  # or "" to skip
```

**Expected outputs:** BLEU, ChrF, WER baselines across all languages and directions.
Results report includes SOTA gap analysis, cross-model comparison tables, and
narrative starters for the Introduction and Results sections.
**Estimated runtime (FULL):** 30–60 min per experiment (GPU).

---

### Paper 2 — LinguoMT-Adapt: Fine-tuning and data scaling

```python
PAPER_MODE        = "adaptation"
DEBUG_MODE        = False
ENABLE_FINETUNING = True
FINETUNING_METHOD = "lora"          # recommended starting point
SCALING_BUDGETS   = [100, 500, 1000, 0]   # 0 = full train set
SOTA_FILE         = ""
```

**Expected outputs:** Before/after fine-tuning metric tables + data scaling curves
showing BLEU/WER vs. training samples. Report discusses diminishing returns and the
budget at which fine-tuning stops improving.
**Estimated runtime (FULL):** 2–4 hours per experiment (GPU, LoRA).

---

### Paper 3 — LinguoMT-Audio: Audio strategy analysis

```python
PAPER_MODE        = "audio"
DEBUG_MODE        = False
ENABLE_FINETUNING = False
SCALING_BUDGETS   = []
SOTA_FILE         = ""
```

**Expected outputs:** Comparison of S2TT (end-to-end), normalised audio, trimmed audio,
and chunk-based audio processing paths. Report highlights which strategy recovers the
most BLEU on noisy or long recordings.
**Estimated runtime (FULL):** 45–90 min per experiment (GPU).

---

### Paper 4 — LinguoMT-Cascade: Cascade vs. end-to-end analysis

```python
PAPER_MODE        = "cascade"
DEBUG_MODE        = False
ENABLE_FINETUNING = False
SCALING_BUDGETS   = []
SOTA_FILE         = ""
```

**Expected outputs:** Oracle cascade analysis (gold transcript → MT), error propagation
curves (BLEU vs. injected WER), and break-even WER thresholds. Report derives the WER
below which cascade matches or beats end-to-end.
**Note:** Run both a SeamlessM4T and a Whisper+NLLB notebook for this paper — you need
the end-to-end baseline (SeamlessM4T) and the cascade scores (Whisper+NLLB) together.
**Estimated runtime (FULL):** 30–60 min per experiment (GPU).

---

### Paper 5 — LinguoMT-Transfer: Cross-lingual transfer

```python
PAPER_MODE        = "transfer"
DEBUG_MODE        = False
ENABLE_FINETUNING = False
SCALING_BUDGETS   = []
SOTA_FILE         = ""
```

**Expected outputs:** Typological similarity scores (lang2vec), cross-lingual transfer
matrices (fine-tune on language A, evaluate on language B), and few-shot scaling curves.
Report discusses which language pairs transfer most effectively and whether typological
distance predicts transfer quality.
**Note:** Use FLEURS notebooks for this paper — African-Celtic does not cover enough
languages for meaningful cross-lingual analysis.
**Estimated runtime (FULL):** 1–2 hours per experiment (GPU).

---

## Complete parameter reference

### `PAPER_MODE` — Which analysis to run

| Value | Paper | What the report covers |
|---|---|---|
| `"benchmark"` | Paper 1 | Zero-shot BLEU/ChrF/WER baselines, SOTA gap analysis |
| `"adaptation"` | Paper 2 | Before/after fine-tuning comparison, data scaling curves |
| `"audio"` | Paper 3 | Audio strategy comparison (S2TT, normalise, trim, chunk) |
| `"cascade"` | Paper 4 | Oracle cascade, error propagation, break-even WER |
| `"transfer"` | Paper 5 | Typological similarity, cross-lingual transfer, few-shot |

**Decision rule:** Set this to match the paper you are writing. The framework always
collects all metrics; `PAPER_MODE` only determines which extra analyses to run and
which report sections to produce.

---

### `DEBUG_MODE` — Speed vs. completeness

| Value | Samples per language | Approx. runtime | Use for |
|---|---|---|---|
| `True` | ~20 | 10–15 min | Verifying the pipeline runs without errors |
| `False` | All | 30 min – 4 h | Paper-quality results |

**Decision rule:** Always start with `DEBUG_MODE = True` to smoke-test the pipeline.
Only switch to `False` when you are confident everything runs correctly.

---

### `ENABLE_FINETUNING` — Fine-tune before evaluation

`False` (default) — evaluate the pretrained model zero-shot. Use for Papers 1, 3, 4, 5.

`True` — fine-tune the model on the training split before evaluating on the dev split.
Use for Papers 2 and 3. Fine-tuning adds significant runtime (1–4 hours).

**When disabled:** `FINETUNING_METHOD` and `SCALING_BUDGETS` are ignored.

---

### `FINETUNING_METHOD` — How parameters are updated

| Value | Description | GPU memory | Speed | Quality |
|---|---|---|---|---|
| `"lora"` | Low-rank adapters only; most parameters frozen | Low (~10 GB) | Fastest | Good |
| `"adapter"` | Bottleneck adapters inserted between layers | Medium (~14 GB) | Medium | Good |
| `"full"` | All parameters updated | High (~24 GB) | Slowest | Best |

**Decision rule:** Start with `"lora"` — it runs on a T4 GPU and achieves most of the
gains of full fine-tuning. Use `"full"` only on A100 if `"lora"` results are insufficient.

---

### `SCALING_BUDGETS` — Training set sizes for data scaling (Paper 2 only)

```python
SCALING_BUDGETS = [100, 500, 1000, 0]   # 0 means the full training set
SCALING_BUDGETS = []                     # disabled — no scaling experiment
```

The framework fine-tunes at each budget value and records metrics, producing a
BLEU-vs-samples curve. `0` always means "use all available training pairs."

**Decision rule:** Only set this for Paper 2. Leave `[]` for all other papers.

---

### `SOTA_FILE` — Compare against published systems (Paper 1 only)

```python
SOTA_FILE = "sota/paper1_benchmark/sota_results.csv"   # relative to repo root
SOTA_FILE = ""   # skip SOTA comparison
```

The CSV must have columns: `system`, `language`, `BLEU`, `venue`, `year`.
Example:

```
system,language,BLEU,venue,year
mSLAM,yoruba,8.4,Interspeech,2022
SeamlessM4T-v1,yoruba,12.1,arXiv,2023
```

**Decision rule:** Provide a SOTA file for Paper 1 to get a gap-analysis table.
Leave `""` for all other papers unless you want the comparison section anyway.

---

### `FORCE_RERUN` — Invalidate the dataset cache

`False` (default) — use the cached dataset if it exists (saves 5–15 min on re-runs).

`True` — re-download and rebuild the dataset cache. Use only if the dataset has
changed or you suspect a corrupted cache.


## Step 1 — Mount Google Drive

Outputs and the results report are backed up to `MyDrive/LinguoMT-AfricaS2T/` at the end of the run.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Drive mounted.")
except ImportError:
    print("Not in Colab — Drive mount skipped.")

## Step 2 — Clone / Update Repository

In [ ]:
import os, subprocess

REPO_DIR = "/content/LinguoMT-AfricaS2T"
REPO_URL = "https://github.com/prsisda/LinguoMT-AfricaS2T.git"

if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)
    print("Repo updated to origin/main")
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print("Repo cloned")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## Step 3 — Install Dependencies

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-U",
    "transformers>=4.40", "datasets", "sacrebleu", "librosa", "soundfile",
    "sentencepiece", "accelerate", "jiwer", "pandas==2.2.2",
    "pyarrow>=15.0.0", "protobuf", "tabulate",
], check=True)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "torchcodec",
    "--extra-index-url", "https://download.pytorch.org/whl/cu121"], check=False)
print("\nDependencies ready.")

## Step 4 — Configure Paper Mode & Settings

| Variable | Values | Notes |
|---|---|---|
| `PAPER_MODE` | `benchmark` · `adaptation` · `audio` · `cascade` · `transfer` | Which paper to produce |
| `DEBUG_MODE` | `True` / `False` | `True` ≈ ~10 min · `False` ≈ ~30–60 min |
| `ENABLE_FINETUNING` | `True` / `False` | Papers 2 & 3 only |
| `SCALING_BUDGETS` | `[100,500,1000,0]` or `[]` | Paper 2 only |
| `SOTA_FILE` | path or `""` | CSV with system/language/BLEU/venue/year columns |

`EXPERIMENT` is fixed to `FLEURS__SeamlessM4Tv2` for this notebook.

In [ ]:
import re, pathlib

# ── SELECT PAPER (uncomment exactly one) ──────────────────────────────
PAPER_MODE = "benchmark"    # Paper 1 — zero-shot baselines           ← ACTIVE
# PAPER_MODE = "adaptation" # Paper 2 — fine-tuning comparison
# PAPER_MODE = "audio"      # Paper 3 — audio strategy analysis
# PAPER_MODE = "cascade"    # Paper 4 — cascade vs end-to-end
# PAPER_MODE = "transfer"   # Paper 5 — cross-lingual transfer
# ─────────────────────────────────────────────────────────────────────

# ── RUN MODE ──────────────────────────────────────────────────────────
DEBUG_MODE = True   # True ≈ ~10 min | False ≈ ~30–60 min
# ─────────────────────────────────────────────────────────────────────

# ── AUDIO DEBUG SKIP ──────────────────────────────────────────────────
# False → audio eval runs in all modes, including DEBUG_MODE.
#         Required for Paper 3 (audio strategies) — True suppresses
#         audio_metrics.csv and audio_metrics_all.csv entirely.
#         Recommended for Paper 2 (adaptation) to capture before/after audio results.
# True  → skips audio evaluation when DEBUG_MODE=True (faster smoke test).
#         Safe for Papers 1, 4, 5 where text/ASR metrics are the primary output
#         and a quick pipeline check does not need audio strategy results.
SKIP_AUDIO_DEBUG = False
# ─────────────────────────────────────────────────────────────────────

# ── FINE-TUNING (Papers 2 & 3 only) ──────────────────────────────────
ENABLE_FINETUNING = False
FINETUNING_METHOD = "lora"   # lora | adapter | full
# ─────────────────────────────────────────────────────────────────────

# ── DATA SCALING BUDGETS (Paper 2 only) ──────────────────────────────
# e.g. SCALING_BUDGETS = [100, 500, 1000, 0]   # 0 = full train set
SCALING_BUDGETS = []
# ─────────────────────────────────────────────────────────────────────

# ── SOTA FILE (optional) ──────────────────────────────────────────────
# e.g. "sota/paper1_benchmark/sota_results.csv"  — leave "" to skip
SOTA_FILE = ""
# ─────────────────────────────────────────────────────────────────────

# Experiment is fixed for this notebook
EXPERIMENT = "FLEURS__SeamlessM4Tv2"
EXPERIMENT_FAMILY = "FLEURS__SeamlessM4Tv2_Large"
SCRIPT  = pathlib.Path(f"{EXPERIMENT}/notebooks/run_experiment.py")
SCRIPTS = {EXPERIMENT: SCRIPT}

print(f"Experiment : {EXPERIMENT}")
print(f"Paper      : {PAPER_MODE}")
print(f"Mode       : {'DEBUG  (fast test)' if DEBUG_MODE else 'FULL   (paper run)'}")
print(f"Audio skip : {SKIP_AUDIO_DEBUG}  (True = no audio_metrics_all.csv in DEBUG_MODE)")
print(f"Fine-tune  : {ENABLE_FINETUNING}  (method: {FINETUNING_METHOD})")
print(f"Scaling    : {SCALING_BUDGETS if SCALING_BUDGETS else 'disabled'}")
print(f"SOTA file  : {SOTA_FILE or 'disabled'}")

## Step 5 — Patch Script & Run Experiment

Pulls the latest framework code, applies your configuration, and runs the `FLEURS__SeamlessM4Tv2` pipeline.

Output is written to `/content/outputs/<timestamp_slug>/`.

In [ ]:
import subprocess, re

subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "reset", "--hard", "origin/main"], check=True)
print("Repository updated.")

mode_str         = "True" if DEBUG_MODE else "False"
ft_str           = "True" if ENABLE_FINETUNING else "False"
budgets_str      = repr(SCALING_BUDGETS)
skip_audio_str   = "False" if not SKIP_AUDIO_DEBUG else "True"

for name, sp in SCRIPTS.items():
    src = sp.read_text()
    src = re.sub(r"(?m)^(DEBUG_MODE\s*=\s*)(True|False)",             rf"\g<1>{mode_str}",           src)
    src = re.sub(r'(?m)^(PAPER_MODE\s*=\s*)["\'][^"\']+["\']',        rf'\g<1>"{PAPER_MODE}"',        src)
    src = re.sub(r"(?m)^(ENABLE_FINETUNING\s*=\s*)(True|False)",      rf"\g<1>{ft_str}",              src)
    src = re.sub(r"(?m)^(FINETUNING_METHOD\s*=\s*)['\"][^'\"]+['\"]", rf"\g<1>'{FINETUNING_METHOD}'", src)
    src = re.sub(r"(?m)^(SCALING_BUDGETS\s*=\s*)\[[^\]]*\]",          rf"\g<1>{budgets_str}",         src)
    src = re.sub(r'(?m)^(SOTA_FILE\s*=\s*)["\'][^"\']*["\']',         rf'\g<1>"{SOTA_FILE}"',         src)
    src = re.sub(r"(?m)^(SKIP_AUDIO_DEBUG\s*=\s*)(True|False)",       rf"\g<1>{skip_audio_str}",      src)
    sp.write_text(src)
    print(f"  Patched: {name}")

print(f"\nScripts configured — {PAPER_MODE} | {'DEBUG' if DEBUG_MODE else 'FULL'} | skip_audio={SKIP_AUDIO_DEBUG}")

In [ ]:
import sys, subprocess as _sp

for name, sp in SCRIPTS.items():
    print(f"\n{'='*64}\n  Running : {name}\n  Paper   : {PAPER_MODE}  |  Mode : {'DEBUG' if DEBUG_MODE else 'FULL'}\n{'='*64}\n")
    result = _sp.run([sys.executable, str(sp)])
    if result.returncode != 0:
        raise RuntimeError(f"{name} failed (exit code {result.returncode})")

print("\nAll experiments finished.")

## Step 6 — Consolidate Metrics

Merges metric CSVs from this experiment's output into a consolidated directory.

In [ ]:
import json as _json
import pandas as pd
from pathlib import Path
from datetime import datetime

out_root         = Path("/content/outputs")
consolidated_dir = out_root / f"consolidated_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}"
consolidated_dir.mkdir(parents=True, exist_ok=True)

all_text, all_asr, all_audio = [], [], []

for run_dir in sorted(out_root.glob("*/")):
    if "consolidated" in run_dir.name:
        continue
    exp_label = run_dir.name
    cfg_path  = run_dir / "config.json"
    if cfg_path.exists():
        exp_label = _json.loads(cfg_path.read_text()).get("experiment_family", run_dir.name)
    for fname, store in [
        ("text_metrics.csv",  all_text),
        ("asr_metrics.csv",   all_asr),
        ("audio_metrics.csv", all_audio),
    ]:
        fpath = run_dir / "metrics" / fname
        if fpath.exists():
            df = pd.read_csv(fpath)
            if "experiment" not in df.columns:
                df.insert(0, "experiment", exp_label)
            store.append(df)

summary_parts = [f"# LinguoMT Consolidated Metrics — {PAPER_MODE}\n\n"]
for label, store, out_name in [
    ("Text Translation", all_text,  "text_metrics_all.csv"),
    ("ASR",              all_asr,   "asr_metrics_all.csv"),
    ("Audio",            all_audio, "audio_metrics_all.csv"),
]:
    if store:
        merged = pd.concat(store, ignore_index=True)
        merged.to_csv(consolidated_dir / out_name, index=False)
        summary_parts += [f"## {label}\n\n", merged.to_markdown(index=False), "\n\n"]
        print(f"=== {label} ===\n{merged.to_string(index=False)}\n")

(consolidated_dir / "metrics_summary.md").write_text("".join(summary_parts))
print(f"Consolidated → {consolidated_dir}")

## Step 7 — Generate Results Report

Reads the consolidated metrics and produces `papers/<paper_id>/results_report.md` — a structured Markdown document with results tables, SOTA comparison, and discussion.

> Results for other experiments in this paper will be absent until you run their notebooks.
> The report shows what is available and marks missing data clearly.

In [ ]:
import subprocess, sys
from pathlib import Path
from pathlib import Path

out_root = Path("/content/outputs")
cons = sorted(out_root.glob("consolidated_*/"), reverse=True)
if not cons:
    print("ERROR: No consolidated directory found — run Step 6 first.")
else:
    consolidated_dir = cons[0]
    cmd = [
        sys.executable, "papers/generate_report.py", PAPER_MODE,
        "--consolidated-dir", str(consolidated_dir),
    ]
    if SOTA_FILE:
        cmd += ["--sota-file", SOTA_FILE]

    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
    else:
        paper_id_map = {
            "benchmark":  "paper1_benchmark",
            "adaptation": "paper2_adaptation",
            "audio":      "paper3_audio",
            "cascade":    "paper4_cascade",
            "transfer":   "paper5_transfer",
        }
        report_path = Path(f"papers/{paper_id_map[PAPER_MODE]}/results_report.md")
        if report_path.exists():
            text    = report_path.read_text()
            preview = text[:5000]
            print("\n" + "="*64)
            print(f"REPORT PREVIEW  —  {report_path}")
            print("="*64)
            print(preview)
            if len(text) > 5000:
                print(f"\n... ({len(text) - 5000} more chars — open the file for the complete report)")

## Step 8 — Package & Download

Zips the report, consolidated metrics, tables, plots, and interpretations. Saves to Drive and triggers download.

In [ ]:
import shutil, json as _json
from pathlib import Path
from datetime import datetime

try:
    from google.colab import files as _colab_files
    _in_colab = True
except ImportError:
    _in_colab = False

ts        = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
repo_root = Path("/content/LinguoMT-AfricaS2T")
mode_tag  = "debug" if DEBUG_MODE else "full"
paper_id_map = {
    "benchmark":  "paper1_benchmark",
    "adaptation": "paper2_adaptation",
    "audio":      "paper3_audio",
    "cascade":    "paper4_cascade",
    "transfer":   "paper5_transfer",
}
PAPER_ID = paper_id_map[PAPER_MODE]
pkg_name  = f"linguomt_{PAPER_ID}_{mode_tag}_{ts}"
pkg_dir   = Path("/content") / pkg_name
pkg_dir.mkdir(parents=True, exist_ok=True)

# 1. Results report (Markdown)
report_md = repo_root / "papers" / PAPER_ID / "results_report.md"
if report_md.exists():
    shutil.copy2(str(report_md), str(pkg_dir / "results_report.md"))

# 2. Consolidated metrics
out_root = Path("/content/outputs")
cons = sorted(out_root.glob("consolidated_*/"), reverse=True)
if cons:
    shutil.copytree(str(cons[0]), str(pkg_dir / "consolidated_metrics"))

# 3. Per-experiment tables, plots, interpretations, summaries
for run_dir in sorted(out_root.glob("*/")):
    if "consolidated" in run_dir.name:
        continue
    cfg_path = run_dir / "config.json"
    label = run_dir.name
    if cfg_path.exists():
        label = _json.loads(cfg_path.read_text()).get("experiment_family", label)
    for sub in ["tables", "plots", "interpretations", "summaries"]:
        src = run_dir / sub
        if src.exists():
            shutil.copytree(str(src), str(pkg_dir / label / sub), dirs_exist_ok=True)

zip_path = shutil.make_archive(f"/content/{pkg_name}", "zip", root_dir=str(pkg_dir))
print(f"Package: {zip_path}")

drive_dir = Path("/content/drive/MyDrive/LinguoMT-AfricaS2T")
if drive_dir.exists():
    drive_dest = drive_dir / f"{pkg_name}.zip"
    shutil.copy2(zip_path, str(drive_dest))
    print(f"Drive backup: {drive_dest}")

if _in_colab:
    _colab_files.download(zip_path)
    print("Download triggered.")
else:
    print(f"Local package: {zip_path}")